In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 物流的发货清单
# 财务的发货清单
# 核算价
# PLM的生命周期全表
### 输出的所有文件
# 统计周期内产品核算价汇总 用于物料精简报告，因为里面有国内国外
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾  ，用于低效-长尾报告，里面只有国内
# 单型号贡献-统计值



### MAP关系汇总1、渠道对照 2、最终表格产品类别对应的产品组集合 3、产品组集合

In [2]:
# 物流的渠道对照关系清洗用
month = 202511
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'无',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'无',
'每誉':'每誉',
'渠道':'无',
'海外':'海外',
'调出渠道':'无',
'非零售工程电商':'非零售工程电商',
'无':'无'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 开票维度

In [3]:
df_income = pd.read_excel(r'C:\Users\zhangbon\Desktop\2024 2025收入.xlsx')
df_product = pd.read_excel(r"D:\000物料报表\202511\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_product['物料号'] = df_product['物料号'].astype(str).str[:13]
df_income['物料编码'] = df_income['物料编码'].astype(str).str[:13]
df_income['年月'] = pd.to_datetime(df_income['年月']).dt.strftime('%Y-%m-01')

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
#只要2024年11月到2025年10月的数据、渠道筛选、产品线筛选、核算价不为0的
df_calu = df_income[(df_income['年月'] >= '2024-11-01') & 
                    (df_income['年月'] <= '2025-10-01') & 
                    (df_income['业务线'].isin(['零售','工程','电商'])) &
                    (df_income['产品线'].isin(['油烟机产品线','烹饪厨电产品线','洗碗机产品线','冰储产品线','净热产品线',])) & 
                    (df_income['核算价金额总计']!=0)].reset_index(drop=True)
df_calu.head()


,年,年月,产品线,任务分类,产品类别,物料编码,物料名称,业务线,数量,不含税收入总计,...,产品类别（手工分类）,系列1,系列2,N代,项目1,项目2,项目上市时间,项目ADCP时间,套系,山头项目
0,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100000,BCD-502WJLPAT,零售,174.0,1.290168e+06,...,家用冰箱,十字,502升,一代,P-2021057-RAAA2001,P-2021057-RAAA2001,2022-04-15,2022-08-31,非套系,0
1,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100000,BCD-502WJLPAT,工程,-1.0,-5.792040e+03,...,家用冰箱,十字,502升,一代,P-2021057-RAAA2001,P-2021057-RAAA2001,2022-04-15,2022-08-31,非套系,0
2,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100000,BCD-502WJLPAT,电商,1.0,7.322871e+03,...,家用冰箱,十字,502升,一代,P-2021057-RAAA2001,P-2021057-RAAA2001,2022-04-15,2022-08-31,非套系,0
3,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100001,BCD-502WYHPAT,零售,194.0,1.459579e+06,...,家用冰箱,十字,502升,一代,P-2024079-一代十字玥影灰项目,P-2024079-一代十字玥影灰项目,2024-11-26,2024-11-26,非套系,0
4,2025,2025-09-01,冰储产品线,冰箱,冰箱,1019000100001,BCD-502WYHPAT,工程,5.0,2.540257e+04,...,家用冰箱,十字,502升,一代,P-2024079-一代十字玥影灰项目,P-2024079-一代十字玥影灰项目,2024-11-26,2024-11-26,非套系,0


In [5]:
# 产品维度——聚合算出每个产品的总销售量和金额
df_group1 = df_calu.groupby('物料编码',as_index=False).agg(
                                                        产品线=('产品线','first'),
                                                        产品总数量=('数量','sum'),
                                                        产品总金额=('核算价金额总计','sum'))
df_group1

,物料编码,产品线,产品总数量,产品总金额
0,1001000100114,油烟机产品线,-1.0,-1900.0
1,1001000200030,油烟机产品线,-13.0,-153400.0
2,1001000200040,油烟机产品线,-1.0,-9000.0
3,1001000200087,油烟机产品线,16.0,310400.0
4,1001000300046,油烟机产品线,-3.0,-6114.0
...,...,...,...,...
1984,1124000202970,洗碗机产品线,6.0,23400.0
1985,1124000202990,洗碗机产品线,8.0,12000.0
1986,1124000203000,洗碗机产品线,8.0,29600.0
1987,1124000204850,洗碗机产品线,1.0,2980.0


In [6]:
# 依据算出的依据物料编码聚合的产品总销售量和金额表，来计算累计销售金额占比、用于找出哪些型号贡献最大
df_out1 = df_group1.sort_values(by='产品总金额', ascending=False).reset_index(drop=True)
df_out1['累计销售金额'] = df_out1['产品总金额'].cumsum()
df_out1['所有产品总销售金额'] = df_out1['产品总金额'].sum()
df_out1['累计销售金额占比'] = df_out1['累计销售金额']/ df_out1['所有产品总销售金额']
df_out1['是否符合2080法则'] = df_out1['累计销售金额占比'] >= 0.8
df_out1


,物料编码,产品线,产品总数量,产品总金额,累计销售金额,所有产品总销售金额,累计销售金额占比,是否符合2080法则
0,1001001500097,油烟机产品线,232850.0,612321372.0,6.123214e+08,1.661581e+10,0.036852,False
1,1001000900395,油烟机产品线,153648.0,382276224.0,9.945976e+08,1.661581e+10,0.059858,False
2,1001002000029,油烟机产品线,100561.0,315764388.0,1.310362e+09,1.661581e+10,0.078862,False
3,1001001500106,油烟机产品线,101710.0,304309872.0,1.614672e+09,1.661581e+10,0.097177,False
4,1002003700004,烹饪厨电产品线,147461.0,255280530.0,1.869952e+09,1.661581e+10,0.112541,False
...,...,...,...,...,...,...,...,...
1984,1001001500032,油烟机产品线,-145.0,-718910.0,1.662054e+10,1.661581e+10,1.000284,True
1985,1005000400002,烹饪厨电产品线,-141.0,-722010.0,1.661982e+10,1.661581e+10,1.000241,True
1986,1005000400020,烹饪厨电产品线,-147.0,-740880.0,1.661908e+10,1.661581e+10,1.000196,True
1987,1019000200003,冰储产品线,-170.0,-1455540.0,1.661762e+10,1.661581e+10,1.000109,True


In [7]:
# df_result1_temp是只留下了贡献了80%收入的型号
product_line = {'油烟机产品线':0,'烹饪厨电产品线':1,'洗碗机产品线':2,'冰储产品线':3,'净热产品线':4,}
i_index1 = df_out1[df_out1['是否符合2080法则']==False].index.max() + 1
df_result1_temp = df_out1.loc[:i_index1]
df_result1_temp

,物料编码,产品线,产品总数量,产品总金额,累计销售金额,所有产品总销售金额,累计销售金额占比,是否符合2080法则
0,1001001500097,油烟机产品线,232850.0,612321372.0,6.123214e+08,1.661581e+10,0.036852,False
1,1001000900395,油烟机产品线,153648.0,382276224.0,9.945976e+08,1.661581e+10,0.059858,False
2,1001002000029,油烟机产品线,100561.0,315764388.0,1.310362e+09,1.661581e+10,0.078862,False
3,1001001500106,油烟机产品线,101710.0,304309872.0,1.614672e+09,1.661581e+10,0.097177,False
4,1002003700004,烹饪厨电产品线,147461.0,255280530.0,1.869952e+09,1.661581e+10,0.112541,False
...,...,...,...,...,...,...,...,...
204,1001000800338,油烟机产品线,9784.0,17493792.0,1.322763e+10,1.661581e+10,0.796087,False
205,1002003400144,烹饪厨电产品线,15194.0,17483390.0,1.324511e+10,1.661581e+10,0.797139,False
206,1009001100015,烹饪厨电产品线,5094.0,17472420.0,1.326258e+10,1.661581e+10,0.798191,False
207,1001000900347,油烟机产品线,9201.0,17371488.0,1.327996e+10,1.661581e+10,0.799236,False


In [8]:
# 依据贡献高的型号表，再看每个产品线各贡献了多少个型号，以及
df_result1 = df_result1_temp.groupby('产品线',as_index=False).agg(                                    
                                                                各产品线主力型号数量 = ('物料编码','nunique')
).sort_values('产品线',key=lambda x:x.map(product_line)).reset_index(drop=True)
df_result1['主力型号总数'] = df_result1['各产品线主力型号数量'].sum()
df_result1['产品型号总数'] = df_group1['物料编码'].nunique()
df_result1['收入总金额'] = df_group1['产品总金额'].sum()
df_result1

,产品线,各产品线主力型号数量,主力型号总数,产品型号总数,收入总金额
0,油烟机产品线,83,209,1989,1.661581e+10
1,烹饪厨电产品线,66,209,1989,1.661581e+10
2,洗碗机产品线,33,209,1989,1.661581e+10
3,冰储产品线,17,209,1989,1.661581e+10
4,净热产品线,10,209,1989,1.661581e+10


In [9]:
# 依据主力型号表，来计算都有哪些生命周期，贡献是多少
df_result1_temp['物料编码'] = df_result1_temp['物料编码'].astype(str).str[:13]
df_result1_temp['生命周期状态'] = df_result1_temp['物料编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))
df_result3 = df_result1_temp.groupby('生命周期状态',as_index=False).agg(
                                                        主力型号数量 = ('物料编码','nunique'),
                                                        各状态主力型号收入总金额 = ('产品总金额','sum')                                                      
)
df_result3['主力型号收入总金额'] = df_result3['各状态主力型号收入总金额'].sum()
df_result3['各状态主力型号收入总金额占比'] = df_result3['各状态主力型号收入总金额']/df_result3['主力型号收入总金额']
df_result3

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_26188\3549675230.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result1_temp['物料编码'] = df_result1_temp['物料编码'].astype(str).str[:13]
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_26188\3549675230.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result1_temp['生命周期状态'] = df_result1_temp['物料编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))


,生命周期状态,主力型号数量,各状态主力型号收入总金额,主力型号收入总金额,各状态主力型号收入总金额占比
0,停止销售,23,8.692572e+08,1.329729e+10,0.065371
1,退市预警,28,3.686898e+09,1.329729e+10,0.277267
2,量产,158,8.741136e+09,1.329729e+10,0.657362


In [10]:
# 产品线-物料编码维度
df_group2 = df_calu.groupby(['产品线','物料编码'],as_index=False).agg(
                                                                    产品总数量=('数量','sum'),
                                                                    产品总金额=('核算价金额总计','sum'),
)
df_group2

,产品线,物料编码,产品总数量,产品总金额
0,冰储产品线,1003000100028,-1.0,-2018.0
1,冰储产品线,1003000100040,-1.0,-2041.0
2,冰储产品线,1003000100046,-1.0,-1685.0
3,冰储产品线,1003000100115,2706.0,9203800.0
4,冰储产品线,1003000100116,2190.0,6504300.0
...,...,...,...,...
1984,烹饪厨电产品线,1023000300052,188.0,1522800.0
1985,烹饪厨电产品线,1023000300054,15.0,105000.0
1986,烹饪厨电产品线,1023000300055,13.0,91000.0
1987,烹饪厨电产品线,1023000300058,45.0,315000.0


In [12]:
# 按照产品线-物料编码维度，计算累计销售金额，用于每个产品线内部判断是否是主力型号
df_out2 = df_group2.sort_values(by=['产品线','产品总金额'], ascending=False).reset_index(drop=True)
df_out2['各产品线累计销售金额'] = df_out2.groupby('产品线')['产品总金额'].transform('cumsum')
df_out2['各产品线所有产品总销售金额'] = df_out2.groupby('产品线')['产品总金额'].transform('sum')
df_out2['各产品线累计销售金额占比'] = df_out2['各产品线累计销售金额'] / df_out2['各产品线所有产品总销售金额']
df_out2['是否符合2080法则'] = df_out2['各产品线累计销售金额占比'] >= 0.8
df_out2

,产品线,物料编码,产品总数量,产品总金额,各产品线累计销售金额,各产品线所有产品总销售金额,各产品线累计销售金额占比,是否符合2080法则
0,烹饪厨电产品线,1002003700004,147461.0,255280530.0,2.552805e+08,5.447020e+09,0.046866,False
1,烹饪厨电产品线,1002003700002,138485.0,239772810.0,4.950533e+08,5.447020e+09,0.090885,False
2,烹饪厨电产品线,1002004300033,139580.0,213557400.0,7.086107e+08,5.447020e+09,0.130091,False
3,烹饪厨电产品线,1002003400077,211137.0,169899220.0,8.785100e+08,5.447020e+09,0.161283,False
4,烹饪厨电产品线,1009000600026,35652.0,142812288.0,1.021322e+09,5.447020e+09,0.187501,False
...,...,...,...,...,...,...,...,...
1984,冰储产品线,1003000200010,-49.0,-89670.0,6.510639e+08,6.473218e+08,1.005781,True
1985,冰储产品线,1003000700001,-64.0,-209920.0,6.508540e+08,6.473218e+08,1.005457,True
1986,冰储产品线,1003000600006,-73.0,-270100.0,6.505839e+08,6.473218e+08,1.005039,True
1987,冰储产品线,1019000200003,-170.0,-1455540.0,6.491283e+08,6.473218e+08,1.002791,True


In [13]:
# 按照产品线维度，计算产品线收入、产品线总型号数、主力型号数
df_result2 = df_out2.groupby('产品线',as_index=False).agg(
                                        产品线收入 = ('产品总金额','sum'),
                                        各产品线总型号数 = ('物料编码','nunique'),
                                        主力型号数 = ('是否符合2080法则',lambda x: (x==False).sum() + 1)
).sort_values(by='产品线',key=lambda x: x.map(product_line)).reset_index(drop=True)
df_result2

,产品线,产品线收入,各产品线总型号数,主力型号数
0,油烟机产品线,7.682855e+09,451,54
1,烹饪厨电产品线,5.447020e+09,892,85
2,洗碗机产品线,2.134049e+09,250,38
3,冰储产品线,6.473218e+08,101,20
4,净热产品线,7.045674e+08,295,39


In [14]:
with pd.ExcelWriter(r'C:\Users\zhangbon\Desktop\单型号贡献_开票维度_1107.xlsx') as writer:
    df_out1.to_excel(writer, sheet_name='所有产品维度', index=False)
    df_out2.to_excel(writer, sheet_name='产品线——产品维度', index=False)
    df_result1.to_excel(writer, sheet_name='所有产品维度_主力型号', index=False)
    df_result2.to_excel(writer, sheet_name='产品线——产品维度_主力型号', index=False)
    df_result3.to_excel(writer, sheet_name='生命周期状态维度_主力型号', index=False)
    

### 物流维度

In [15]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2025年10月明细.xlsx', '10月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年10月明细.xlsx'), ('2025年1月明细.xlsx', '1月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年1月明细.xlsx'), ('2025年2月明细.xlsx', '2月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年2月明细.xlsx'), ('2025年3月明细.xlsx', '3月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年3月明细.xlsx'), ('2025年4月明细.xlsx', '4月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年4月明细.xlsx'), ('2025年5月明细.xlsx', '5月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年5月明细.xlsx'), ('2025年6月明细.xlsx', '6月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年6月明细.xlsx'), ('2025年7月明细.xlsx', '7月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年7月明细.xlsx'), ('2025年8月明细.xlsx', '8月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年8月明细.xlsx'), ('2025年9月明细.xlsx', '9月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年9月明细.xlsx')]


In [16]:
#如果有报错请提示报错信息
df = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量']]
    df = pd.concat([df, df_temp], axis=0).reset_index(drop=True)
df = df.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df['商品编码'] = df['商品编码'].astype(str)
df['渠道'].value_counts()

成功读取2025年10月明细.xlsx的10月明细表
成功读取2025年1月明细.xlsx的1月明细表
成功读取2025年2月明细.xlsx的2月明细表
成功读取2025年3月明细.xlsx的3月明细表
成功读取2025年4月明细.xlsx的4月明细表
成功读取2025年5月明细.xlsx的5月明细表
成功读取2025年6月明细.xlsx的6月明细表
成功读取2025年7月明细.xlsx的7月明细表
成功读取2025年8月明细.xlsx的8月明细表
成功读取2025年9月明细.xlsx的9月明细表


渠道
零售        390588
工程         16940
电商          7855
电商不可售       6616
海外          1771
每誉           152
战略电商         129
新品            67
内部处理通用        15
借出渠道          14
商净             7
渠道             4
调出渠道           2
转出渠道           2
米博新零售          1
Name: count, dtype: int64

In [17]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\财务发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2024年11月明细.xlsx', '11月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\财务发货\\2024年11月明细.xlsx'), ('2024年12月明细.xlsx', '12月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\财务发货\\2024年12月明细.xlsx')]


In [18]:
caiwu_shouru = {'商品编码':[],'渠道':[],'实际出库数量':[]}
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(file[0], file[1])
    for index,row in df_temp.iterrows():
        if row['零售'] > 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('零售')
            caiwu_shouru['实际出库数量'].append(row['零售'])
        if row['工程'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('工程')
            caiwu_shouru['实际出库数量'].append(row['工程'])
        if row['电商'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('电商')
            caiwu_shouru['实际出库数量'].append(row['电商'])
        if row['合计-发货'] - row['零售'] - row['工程'] - row['电商'] != 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('非零售工程电商')
            caiwu_shouru['实际出库数量'].append(row['合计-发货'] - row['零售'] - row['工程'] - row['电商'])

df_caiwu = pd.DataFrame(caiwu_shouru)
df_caiwu['商品编码'] = df_caiwu['商品编码'].astype(str)
df_caiwu['渠道'].value_counts()

成功读取2024年11月明细.xlsx的11月明细表
成功读取2024年12月明细.xlsx的12月明细表


渠道
零售         1029
电商          797
工程          476
非零售工程电商      16
Name: count, dtype: int64

In [19]:
df0 = pd.concat([df, df_caiwu], axis=0).reset_index(drop=True)
df0['商品编码'] = df0['商品编码'].astype(str).str[:13]
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[df0['渠道'].isin(['零售','工程','电商'])].reset_index(drop=True)
df0.info()
# df0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 424512 entries, 0 to 424511
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   商品编码    424512 non-null  object
 1   渠道      424512 non-null  object
 2   实际出库数量  424512 non-null  object
dtypes: object(3)
memory usage: 9.7+ MB


In [20]:
df1 = df0.copy()
df1[['商品编码','渠道']] = df1[['商品编码','渠道']].astype(str)
df1['实际出库数量'] = df1['实际出库数量'].astype(float)

# PLM产品数据导入
df_product_group = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_product_group[['物料号','标准型号','国内/海外','产品线']] = df_product_group[['物料号','标准型号','国内/海外','产品线']].astype(str)
df_product_group = df_product_group[df_product_group['物料号'].str.len()>10]
df_product_group['物料号'] = df_product_group['物料号'].apply(lambda x: x[:13])
# df_product_group

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [21]:
#产品组对照关系
product_group_map = dict(zip(df_product_group['物料号'],df_product_group['产品组']))
#标准型号对照关系
product_standard = dict(zip(df_product_group['物料号'],df_product_group['标准型号']))
#记录国内/海外状态
product_country = dict(zip(df_product_group['物料号'],df_product_group['国内/海外']))
#记录产品线
product_line = dict(zip(df_product_group['物料号'],df_product_group['产品线'])) 


In [22]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['产品编码'] = df_price['产品编码'].astype(str)
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
product_price_map = dict(zip(df_price['产品编码'],df_price['系统核算价']))

In [23]:
df1['产品组'] = df1['商品编码'].map(product_group_map)
df1['产品线'] = df1['商品编码'].map(product_line)
df1['系统核算价'] = df1['商品编码'].map(product_price_map)
df1['核算价'] = df1['系统核算价'] * df1['实际出库数量']
df1['标准型号'] = df1['商品编码'].map(product_standard)
df1['国内/海外'] = df1['商品编码'].map(product_country)
df1 = df1[df1['商品编码'].str.startswith('10')].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'国内没有系统核算价的是这些数据\n{df1[(df1['系统核算价'].isnull()&(df1['国内/海外']=='国内'))]['商品编码'].drop_duplicates()}')
print(len(df1))
df1

国内没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
419684


,商品编码,渠道,实际出库数量,产品组,产品线,系统核算价,核算价,标准型号,国内/海外
0,1001001500116,零售,12.0,吸油烟机,油烟机产品线,3358.0,40296.0,Z8T,国内
1,1009000600033,零售,1.0,蒸烤烹饪机,烹饪厨电产品线,3450.0,3450.0,ZK50-02-F1,国内
2,1009000500035,零售,3.0,灶蒸烤烹饪机,烹饪厨电产品线,5280.0,15840.0,JZT-ZK46-X2,国内
3,1001001500131,零售,6.0,吸油烟机,油烟机产品线,2988.0,17928.0,02-Z6TA,国内
4,1002003700049,零售,5.0,灶具,烹饪厨电产品线,2550.0,12750.0,H8B,国内
...,...,...,...,...,...,...,...,...,...
419679,1019000300007,工程,2.0,家用冰箱,冰储产品线,26923.0,53846.0,BCD-508W-Y1Pro,国内
419680,1019000200004,零售,130.0,家用冰箱,冰储产品线,8562.0,1113060.0,BCD-510WYYPAF,国内
419681,1019000200001,零售,15.0,家用冰箱,冰储产品线,8562.0,128430.0,BCD-510WZBPAF,国内
419682,1019000200001,电商,2.0,家用冰箱,冰储产品线,8562.0,17124.0,BCD-510WZBPAF,国内


In [24]:
df_cleaned = df1[(df1[r'国内/海外']=='国内')&
                  (df1['渠道'].isin(['零售','工程','电商']))&
                (df1['产品线'].isin(['油烟机产品线','烹饪厨电产品线','洗碗机产品线','冰储产品线','净热产品线',]))
    ].reset_index(drop=True)
df_cleaned

,商品编码,渠道,实际出库数量,产品组,产品线,系统核算价,核算价,标准型号,国内/海外
0,1001001500116,零售,12.0,吸油烟机,油烟机产品线,3358.0,40296.0,Z8T,国内
1,1009000600033,零售,1.0,蒸烤烹饪机,烹饪厨电产品线,3450.0,3450.0,ZK50-02-F1,国内
2,1009000500035,零售,3.0,灶蒸烤烹饪机,烹饪厨电产品线,5280.0,15840.0,JZT-ZK46-X2,国内
3,1001001500131,零售,6.0,吸油烟机,油烟机产品线,2988.0,17928.0,02-Z6TA,国内
4,1002003700049,零售,5.0,灶具,烹饪厨电产品线,2550.0,12750.0,H8B,国内
...,...,...,...,...,...,...,...,...,...
412555,1019000300007,工程,2.0,家用冰箱,冰储产品线,26923.0,53846.0,BCD-508W-Y1Pro,国内
412556,1019000200004,零售,130.0,家用冰箱,冰储产品线,8562.0,1113060.0,BCD-510WYYPAF,国内
412557,1019000200001,零售,15.0,家用冰箱,冰储产品线,8562.0,128430.0,BCD-510WZBPAF,国内
412558,1019000200001,电商,2.0,家用冰箱,冰储产品线,8562.0,17124.0,BCD-510WZBPAF,国内


In [25]:
# 产品维度，计算出每个产品的销售总数量和总金额
df_group3 = df_cleaned.groupby('商品编码',as_index=False).agg(
                                                            产品线=('产品线','first'),
                                                            产品总数量=('实际出库数量','sum'),
                                                            产品总金额=('核算价','sum'),
)
df_group3

,商品编码,产品线,产品总数量,产品总金额
0,1001000200087,油烟机产品线,26.0,504400.0
1,1001000300073,油烟机产品线,88.0,194304.0
2,1001000500073,油烟机产品线,148.0,422984.0
3,1001000500224,油烟机产品线,4938.0,14705364.0
4,1001000500244,油烟机产品线,9.0,30492.0
...,...,...,...,...
1146,1019000300005,冰储产品线,2605.0,28628950.0
1147,1019000300006,冰储产品线,2172.0,31029192.0
1148,1019000300007,冰储产品线,100.0,2692300.0
1149,1024000200018,洗碗机产品线,2125.0,5100000.0


In [26]:
# 对产品维度的表进行金额降序排序再进行计算累计求和金额，用于判断主力型号
df_out3 = df_group3.sort_values(by='产品总金额',ascending=False).reset_index(drop=True)
df_out3['累计销售金额'] = df_out3['产品总金额'].cumsum()
df_out3['所有产品总销售金额'] = df_out3['产品总金额'].sum()
df_out3['累计销售金额占比'] = df_out3['累计销售金额']/df_out3['所有产品总销售金额']
df_out3['是否符合2080法则'] = df_out3['累计销售金额占比'] >= 0.8
df_out3

,商品编码,产品线,产品总数量,产品总金额,累计销售金额,所有产品总销售金额,累计销售金额占比,是否符合2080法则
0,1001001500097,油烟机产品线,234620.0,616581360.0,6.165814e+08,1.641475e+10,0.037563,False
1,1001000900395,油烟机产品线,136361.0,339266168.0,9.558475e+08,1.641475e+10,0.058231,False
2,1001002000029,油烟机产品线,103043.0,323348934.0,1.279196e+09,1.641475e+10,0.077930,False
3,1001001500106,油烟机产品线,103167.0,308262996.0,1.587459e+09,1.641475e+10,0.096709,False
4,1002003700004,烹饪厨电产品线,150729.0,260761170.0,1.848221e+09,1.641475e+10,0.112595,False
...,...,...,...,...,...,...,...,...
1146,1002004300071,烹饪厨电产品线,1.0,1400.0,1.641474e+10,1.641475e+10,1.000000,True
1147,1002004300036,烹饪厨电产品线,1.0,1260.0,1.641475e+10,1.641475e+10,1.000000,True
1148,1002003400065,烹饪厨电产品线,1.0,1180.0,1.641475e+10,1.641475e+10,1.000000,True
1149,1002003400169,烹饪厨电产品线,1.0,1080.0,1.641475e+10,1.641475e+10,1.000000,True


In [27]:
# 依据产品维度把主力型号给筛选出来
i_index3 = df_out3[df_out3['是否符合2080法则']==True].index.min()
df_result2_temp = df_out3.loc[:i_index3]
df_result2_temp


,商品编码,产品线,产品总数量,产品总金额,累计销售金额,所有产品总销售金额,累计销售金额占比,是否符合2080法则
0,1001001500097,油烟机产品线,234620.0,616581360.0,6.165814e+08,1.641475e+10,0.037563,False
1,1001000900395,油烟机产品线,136361.0,339266168.0,9.558475e+08,1.641475e+10,0.058231,False
2,1001002000029,油烟机产品线,103043.0,323348934.0,1.279196e+09,1.641475e+10,0.077930,False
3,1001001500106,油烟机产品线,103167.0,308262996.0,1.587459e+09,1.641475e+10,0.096709,False
4,1002003700004,烹饪厨电产品线,150729.0,260761170.0,1.848221e+09,1.641475e+10,0.112595,False
...,...,...,...,...,...,...,...,...
210,1009001400000,烹饪厨电产品线,5367.0,17244171.0,1.307495e+10,1.641475e+10,0.796537,False
211,1001000800338,油烟机产品线,9501.0,16987788.0,1.309194e+10,1.641475e+10,0.797572,False
212,1002003400144,烹饪厨电产品线,14699.0,16903850.0,1.310884e+10,1.641475e+10,0.798601,False
213,1002003500062,烹饪厨电产品线,7281.0,16891920.0,1.312573e+10,1.641475e+10,0.799631,False


In [28]:
# 依据产品维度的主力型号表，把主力型号总数、以及在各个产品线的分布计算出来
product_line = {'油烟机产品线':0,'烹饪厨电产品线':1,'洗碗机产品线':2,'冰储产品线':3,'净热产品线':4,}
df_result4 = df_result2_temp.groupby('产品线',as_index=False).agg(
    各产品线主力型号数量=('商品编码','nunique')
).sort_values('产品线',key=lambda x: x.map(product_line)).reset_index(drop=True)
df_result4['主力型号总数'] = df_result4['各产品线主力型号数量'].sum()
df_result4['产品型号总数'] = df_group3['商品编码'].nunique()
df_result4['收入总金额'] = df_group3['产品总金额'].sum()
df_result4

,产品线,各产品线主力型号数量,主力型号总数,产品型号总数,收入总金额
0,油烟机产品线,84,215,1151,1.641475e+10
1,烹饪厨电产品线,69,215,1151,1.641475e+10
2,洗碗机产品线,36,215,1151,1.641475e+10
3,冰储产品线,16,215,1151,1.641475e+10
4,净热产品线,10,215,1151,1.641475e+10


In [29]:
# 依据主力型号表，计算出生命周期状态分布以及每个状态的金额占比
df_result2_temp['商品编码'] = df_result2_temp['商品编码'].astype(str).str[:13]
df_result2_temp['生命周期状态'] = df_result2_temp['商品编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))
df_result6 = df_result2_temp.groupby('生命周期状态',as_index=False).agg(
                                                        主力型号数量 = ('商品编码','nunique'),
                                                        各状态主力型号收入总金额 = ('产品总金额','sum')                                                      
)
df_result6['主力型号收入总金额'] = df_result6['各状态主力型号收入总金额'].sum()
df_result6['各状态主力型号收入总金额占比'] = df_result6['各状态主力型号收入总金额']/df_result6['主力型号收入总金额']
df_result6


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_26188\3744389675.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result2_temp['商品编码'] = df_result2_temp['商品编码'].astype(str).str[:13]
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_26188\3744389675.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result2_temp['生命周期状态'] = df_result2_temp['商品编码'].map(dict(zip(df_product['物料号'], df_product['产品状态'])))


,生命周期状态,主力型号数量,各状态主力型号收入总金额,主力型号收入总金额,各状态主力型号收入总金额占比
0,停止销售,18,7.226839e+08,1.314258e+10,0.054988
1,退市预警,28,3.555649e+09,1.314258e+10,0.270544
2,量产,169,8.864246e+09,1.314258e+10,0.674468


In [30]:
# 按照产品线-商品编码进行分组，计算出各个产品线下每个产品的总数量和总金额
df_group4 = df_cleaned.groupby(['产品线','商品编码'],as_index=False).agg(
                                                            产品总数量=('实际出库数量','sum'),
                                                            产品总金额=('核算价','sum'),
)
df_group4


,产品线,商品编码,产品总数量,产品总金额
0,冰储产品线,1003000100115,2653.0,9020200.0
1,冰储产品线,1003000100116,2212.0,6569640.0
2,冰储产品线,1003000100117,1161.0,3877740.0
3,冰储产品线,1003000300021,132.0,233640.0
4,冰储产品线,1003000300022,12669.0,19256880.0
...,...,...,...,...
1146,烹饪厨电产品线,1009001500001,7294.0,36834700.0
1147,烹饪厨电产品线,1009001500002,12507.0,63160350.0
1148,烹饪厨电产品线,1009001500005,737.0,5439060.0
1149,烹饪厨电产品线,1009001500006,2017.0,11496900.0


In [31]:
# 按照产品线和产品总金额排序，然后计算各个产品线内的累计金额和占比，用于筛选每个产品线内的主力型号
df_out4 = df_group4.sort_values(by=['产品线','产品总金额'],ascending=False).reset_index(drop=True)
df_out4['累计销售金额'] = df_out4.groupby('产品线')['产品总金额'].cumsum()
df_out4['所有产品总销售金额'] = df_out4.groupby('产品线')['产品总金额'].transform('sum')
df_out4['累计销售金额占比'] = df_out4['累计销售金额']/df_out4['所有产品总销售金额']
df_out4['是否符合2080法则'] = df_out4['累计销售金额占比'] >= 0.8
df_out4


,产品线,商品编码,产品总数量,产品总金额,累计销售金额,所有产品总销售金额,累计销售金额占比,是否符合2080法则
0,烹饪厨电产品线,1002003700004,150729.0,260761170.0,2.607612e+08,5.452440e+09,0.047825,False
1,烹饪厨电产品线,1002003700002,139264.0,240926720.0,5.016879e+08,5.452440e+09,0.092012,False
2,烹饪厨电产品线,1002004300033,123847.0,189485910.0,6.911738e+08,5.452440e+09,0.126764,False
3,烹饪厨电产品线,1002003800013,99131.0,168522700.0,8.596965e+08,5.452440e+09,0.157672,False
4,烹饪厨电产品线,1002003400077,206319.0,162992010.0,1.022689e+09,5.452440e+09,0.187565,False
...,...,...,...,...,...,...,...,...
1146,冰储产品线,1003000500017,46.0,81880.0,6.986208e+08,6.987862e+08,0.999763,True
1147,冰储产品线,1003000500028,30.0,60000.0,6.986808e+08,6.987862e+08,0.999849,True
1148,冰储产品线,1003000300039,41.0,53300.0,6.987341e+08,6.987862e+08,0.999925,True
1149,冰储产品线,1003000900005,12.0,43200.0,6.987773e+08,6.987862e+08,0.999987,True


In [32]:
# 依据产品线-商品编码的金额汇总表，计算每个产品线的收入，以及产品型号的数量、还有主力型号的数量
df_result5 = df_out4.groupby('产品线',as_index=False).agg(
    产品线收入 = ('产品总金额','sum'),
    各产品线总型号数 = ('商品编码','nunique'),
    各产品线主力型号数 = ('是否符合2080法则',lambda x:(x==False).sum() + 1)
).sort_values(by='产品线',key = lambda x:x.map(product_line)).reset_index(drop=True)
df_result5

,产品线,产品线收入,各产品线总型号数,各产品线主力型号数
0,油烟机产品线,7.373208e+09,254,56
1,烹饪厨电产品线,5.452440e+09,541,85
2,洗碗机产品线,2.188430e+09,154,39
3,冰储产品线,6.987862e+08,65,21
4,净热产品线,7.018852e+08,137,38


In [33]:
with pd.ExcelWriter(r'C:\Users\zhangbon\Desktop\单型号贡献_发货维度.xlsx') as writer:
    df_out3.to_excel(writer, sheet_name='产品维度', index=False)
    df_out4.to_excel(writer, sheet_name='产品线产品维度', index=False)
    df_result4.to_excel(writer, sheet_name='所有产品维度_主力型号', index=False)
    df_result5.to_excel(writer, sheet_name='产品线-产品维度_主力型号', index=False)
    df_result6.to_excel(writer, sheet_name='生命周期状态维度_主力型号', index=False)
